# 무무 · 한국어 구어체 Whisper 파인튜닝
[AI-Hub 자유대화 음성(109)](https://aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&dataSetSn=109) 일부로 Whisper small을 학습하고, 같은 평가 음성에서 원본 small과 비교한다. [참고 글](https://ysg2997.tistory.com/53) · [Whisper 학습 가이드](https://huggingface.co/blog/fine-tune-whisper)

**실행 전** Colab에서 GPU 런타임을 선택한다. 이 실험은 AI-Hub의 샘플 ZIP으로 재현할 수 있다.

Google Drive의 내 드라이브에 AI-Hub 샘플 ZIP `New_Sample.zip`을 둔다. 패키지 설치 다음에 Drive를 마운트하면 새 런타임에서도 같은 ZIP을 사용할 수 있다. 아래 셀에서 ZIP을 풀고 WAV와 JSON을 매칭한다. 이 샘플은 사전 점검과 첫 학습 실험에 쓰고, 화자가 더 많은 원본 데이터에서도 같은 방식으로 재평가한다.

In [ ]:
# 패키지가 이미 import되어 있었다면 설치 뒤 런타임을 재시작하세요.

%pip -q install -U "transformers==5.23.0" "huggingface-hub>=1.32,<3" "accelerate>=1.1,<2" "jiwer>=4,<5" soundfile scipy

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# WAV와 JSON을 각각 하위 폴더에 두어도 된다. 압축을 푼 공통 상위 폴더를 지정한다.
DATA_ROOT = Path("/content/aihub109")
ARCHIVE_PATH = Path("/content/drive/MyDrive/New_Sample.zip")
OUTPUT_DIR = Path("/content/mumu-whisper")


In [ ]:
MODEL_ID = "openai/whisper-small"
TRAIN_HOURS = 2.0
MAX_EVAL_CLIPS = 120
SEED = 42

In [ ]:
from pathlib import PurePosixPath
from shutil import copyfileobj
from zipfile import ZipFile

def unpack_sample(archive, target):
    if not archive.is_file():
        if target.is_dir():
            print(f"이미 풀어 둔 파일을 사용합니다: {target}")
            return
        raise FileNotFoundError(f"Google Drive를 마운트하고 ZIP 위치를 확인해 주세요. 찾는 경로: {archive}")

    with ZipFile(archive, metadata_encoding="cp949") as package:
        files, destinations = [], set()
        for item in package.infolist():
            path = PurePosixPath(item.filename)
            if item.is_dir() or path.suffix.lower() not in {".wav", ".json"}:
                continue
            if path.is_absolute() or ".." in path.parts or any(":" in part or chr(92) in part for part in path.parts):
                raise ValueError(f"ZIP 안의 파일 경로를 확인해 주세요: {path}")
            folder = "audio" if path.suffix.lower() == ".wav" else "labels"
            output_path = target / folder / path.name
            flat_key = (folder, path.name.casefold())
            if flat_key in destinations:
                raise ValueError(f"ZIP에 이름이 같은 파일이 있습니다: {path.name}")
            destinations.add(flat_key)
            files.append((item, output_path))
        if not files:
            raise ValueError("ZIP에 WAV 또는 JSON 파일이 없습니다.")

        for item, destination in files:
            if destination.is_file() and destination.stat().st_size == item.file_size:
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with package.open(item) as source, destination.open("wb") as output:
                copyfileobj(source, output)
        print(f"{len(files):,}개 음성·라벨 파일을 준비했습니다: {target}")

unpack_sample(ARCHIVE_PATH, DATA_ROOT)

## 1. WAV와 전사 매칭
AI-Hub 설명서의 `fileNm`, `stt`, `recorderId`를 찾는다. 파일명이 겹치면 자동 매칭에서 제외한다. 화자 ID를 확인할 수 없는 발화는 학습·평가 분리에 넣지 않으므로 건수가 많으면 라벨 JSON 구조를 먼저 확인한다.

In [ ]:
from collections import Counter, defaultdict
import json
import re
import unicodedata
import soundfile as sf

def values_for(node, key):
    if isinstance(node, dict):
        for name, value in node.items():
            if name == key and isinstance(value, (str, int)) and not isinstance(value, bool):
                yield str(value).strip()
            if isinstance(value, (dict, list)):
                yield from values_for(value, key)
    elif isinstance(node, list):
        for value in node:
            yield from values_for(value, key)

def only_value(node, key):
    values = {v for v in values_for(node, key) if v}
    return next(iter(values)) if len(values) == 1 else None

def clean_label(text):
    # (철자)/(발음) 이중 전사는 정답으로 철자 표기를 사용한다.
    text = re.sub(r"\(([^()]*)\)\s*/\s*\([^()]*\)", r"\1", text)
    return " ".join(unicodedata.normalize("NFC", text).split())

def load_pairs(root):
    if not root.is_dir():
        raise FileNotFoundError(f"{root}에 압축을 푼 WAV와 JSON을 두거나 DATA_ROOT를 바꿔 주세요.")
    audio_by_stem = defaultdict(list)
    labels = []
    for path in sorted(root.rglob("*")):
        if path.suffix.lower() == ".wav":
            audio_by_stem[path.stem.casefold()].append(path)
        elif path.suffix.lower() == ".json":
            labels.append(path)

    rows, counts, seen = [], Counter(), set()
    for path in labels:
        try:
            label = json.loads(path.read_text(encoding="utf-8-sig"))
        except (ValueError, OSError):
            counts["읽을 수 없는 JSON"] += 1
            continue
        ref = only_value(label, "stt")
        if not ref:
            counts["전사 없음 또는 여러 전사"] += 1
            continue
        filename = only_value(label, "fileNm")
        stem = Path(filename.replace("\\", "/")).stem if filename else path.stem
        matches = audio_by_stem.get(stem.casefold(), [])
        if len(matches) != 1 or matches[0] in seen:
            counts["WAV 매칭 실패 또는 중복"] += 1
            continue
        try:
            duration = sf.info(matches[0]).duration
        except (sf.LibsndfileError, OSError):
            counts["오디오 읽기 실패"] += 1
            continue
        if not 0 < duration <= 30:
            counts["Whisper 학습에 맞지 않는 길이"] += 1
            continue
        text = clean_label(ref)
        if not any(char.isalnum() for char in text):
            counts["빈 전사"] += 1
            continue
        seen.add(matches[0])
        rows.append(dict(audio=matches[0], text=text,
                         speaker=only_value(label, "recorderId"), duration=duration))
    return rows, counts

rows, skipped = load_pairs(DATA_ROOT)
print(f"음성·전사 {len(rows):,}쌍 / 건너뛴 사유 {dict(skipped)}")
print(f"화자 ID 없음 {sum(not row['speaker'] for row in rows):,}쌍")
for row in rows[:5]:
    print(row["audio"].name, round(row["duration"], 2), row["text"])

In [ ]:
from random import Random

def split_speakers(rows, train_hours, max_eval_clips, seed):
    groups = defaultdict(list)
    for row in rows:
        if row["speaker"]:
            groups[row["speaker"]].append(row)
    speakers = sorted(groups)
    if len(speakers) < 3:
        raise ValueError("화자 ID를 가진 화자 3명 이상이 필요합니다. 경량 샘플에 ID가 없으면 원본 데이터를 준비해 주세요.")
    rng = Random(seed)
    rng.shuffle(speakers)
    eval_count = max(1, round(len(speakers) * 0.2))
    test_speakers = set(speakers[:eval_count])
    test = [row for speaker in speakers[:eval_count] for row in groups[speaker]]
    rng.shuffle(test)
    test = test[:max_eval_clips]

    candidates = [row for speaker in speakers[eval_count:] for row in groups[speaker]]
    rng.shuffle(candidates)
    train, seconds = [], 0.0
    for row in candidates:
        if seconds + row["duration"] <= train_hours * 3600:
            train.append(row)
            seconds += row["duration"]
    if not train or not test:
        raise ValueError("학습 또는 평가 음성이 비었습니다. 데이터 범위와 설정을 확인해 주세요.")
    assert all(row["speaker"] not in test_speakers for row in train)
    return train, test

train_rows, test_rows = split_speakers(rows, TRAIN_HOURS, MAX_EVAL_CLIPS, SEED)
for name, subset in (("학습", train_rows), ("평가", test_rows)):
    print(f"{name}: {len(subset):,}쌍, {len({r['speaker'] for r in subset})}명, "
          f"{sum(r['duration'] for r in subset) / 3600:.2f}시간")

## 2. 원본 Whisper small 평가
두 모델에서 한국어 전사, 동일한 생성 옵션과 평가 파일을 사용한다. CER은 NFC로 정규화한 뒤 띄어쓰기와 문장부호를 제외한다. 이 실험의 평가 음성은 AI-Hub 자유대화이므로 무무 앱의 실사용 정확도와는 별도로 해석한다.

이 경량 ZIP은 4명의 발화로 구성돼 있다. 기본 분리에서는 1명이 평가에만 들어가므로, 전체 사용자에 대한 성능으로 일반화하기 어렵다.

In [ ]:
from math import gcd
import jiwer
from scipy.signal import resample_poly
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형에서 GPU를 선택한 뒤 처음부터 실행해 주세요.")
set_seed(SEED)

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to("cuda")
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"

def read_audio(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if rate != 16000:
        divisor = gcd(rate, 16000)
        audio = resample_poly(audio, 16000 // divisor, rate // divisor)
    return audio

def normalize_cer(text):
    return "".join(ch for ch in unicodedata.normalize("NFC", text).casefold() if ch.isalnum())

@torch.inference_mode()
def predict(model, rows, batch_size=4):
    model.eval()
    predictions = []
    for start in range(0, len(rows), batch_size):
        audio = [read_audio(row["audio"]) for row in rows[start:start + batch_size]]
        inputs = processor.feature_extractor(
            audio, sampling_rate=16000, return_attention_mask=True, return_tensors="pt"
        ).to(model.device)
        ids = model.generate(**inputs, language="ko", task="transcribe",
                             max_new_tokens=225, num_beams=5, do_sample=False)
        predictions.extend(processor.batch_decode(ids, skip_special_tokens=True))
    return predictions

references = [row["text"] for row in test_rows]
def cer(predictions):
    return jiwer.cer([normalize_cer(ref) for ref in references],
                     [normalize_cer(pred) for pred in predictions])

baseline_predictions = predict(model, test_rows)
print(f"원본 small CER: {cer(baseline_predictions):.2%}")

In [ ]:
from torch.utils.data import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

class SpeechDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        features = processor.feature_extractor(read_audio(row["audio"]), sampling_rate=16000)
        return dict(input_features=features.input_features[0],
                    labels=processor.tokenizer(row["text"]).input_ids)

def collate(items):
    batch = processor.feature_extractor.pad(
        [{"input_features": item["input_features"]} for item in items], return_tensors="pt"
    )
    labels = processor.tokenizer.pad(
        [{"input_ids": item["labels"]} for item in items], return_tensors="pt"
    )
    targets = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
    if (targets[:, 0] == model.config.decoder_start_token_id).all().item():
        targets = targets[:, 1:]
    batch["labels"] = targets
    return batch

model.config.use_cache = False
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=1,
    gradient_checkpointing=True,
    fp16=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=SpeechDataset(train_rows),
                         data_collator=collate, processing_class=processor)
trainer.train()

In [ ]:
import shutil

model.gradient_checkpointing_disable()
model.config.use_cache = True
tuned_predictions = predict(model, test_rows)
print(f"원본 small CER: {cer(baseline_predictions):.2%}")
print(f"파인튜닝 small CER: {cer(tuned_predictions):.2%}")

changed = [i for i, (before, after) in enumerate(zip(baseline_predictions, tuned_predictions))
           if normalize_cer(before) != normalize_cer(after)]
for i in changed[:8]:
    print(f"정답  {references[i]}")
    print(f"기존  {baseline_predictions[i]}")
    print(f"학습  {tuned_predictions[i]}")

model_dir = OUTPUT_DIR / "model"
trainer.save_model(str(model_dir))
processor.save_pretrained(model_dir)
archive = shutil.make_archive(str(OUTPUT_DIR / "mumu-whisper-small"), "zip", model_dir)
print(f"로컬 추론용 Transformers 모델: {archive}")

## 3. 결과 보관
Colab 파일 패널에서 생성된 모델 ZIP을 세션 종료 전에 내려받는다. 이 폴더는 Transformers 형식이므로 기존 MUMU의 `whisper.load_model("base")` 호출에 그대로 넣는 대신 Transformers 추론 경로로 연결해야 한다. 평가 수치는 데이터 범위와 화자 수를 함께 기록한다.